# 2.2 Text preprocessing

Guideline notebook `text_preprocessing.ipynb`.

This notebook **does not rerun** the cleaner. It loads `outputs/preprocessing_examples.csv` (36 stored before/after rows).

Two saved columns exist in SQLite `nlp_posts` (database excluded from submission):
- `clean_text` — lowercase; strip URLs, @mentions, emojis, punctuation; drop stopwords; Porter stem. **Keeps** IFB / follow tokens for sentiment.
- `topic_text` — same cleaning, then **removes** IFB / follow / followback / f4f so BERTopic cannot collapse.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

def find_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "outputs").is_dir() and (p / "src").is_dir():
            return p
    raise FileNotFoundError(
        "Cannot find the project root (need outputs/ and src/). "
        "Open this notebook from the repo, from notebooks/, or from a guideline folder."
    )

ROOT = find_root()
OUT = ROOT / "outputs"
print("ROOT =", ROOT)
print("This notebook reads shipped CSVs and PNGs only.")
print("It does not open IFB.db, does not open any analysis database,")
print("and does not rerun sentiment, emotion, BERTopic, NER, network, or bot models.")


ROOT = C:\Users\yosef\OneDrive\שולחן העבודה\IFB-project
This notebook reads shipped CSVs and PNGs only.
It does not open IFB.db, does not open any analysis database,
and does not rerun sentiment, emotion, BERTopic, NER, network, or bot models.


In [2]:
ex = pd.read_csv(OUT / "preprocessing_examples.csv", encoding="utf-8-sig")
print("rows", len(ex))
print("columns", list(ex.columns))
cols = [c for c in ["original_text", "clean_text", "topic_text"] if c in ex.columns]
display(ex[cols].head(12))
# IFB tokens should remain in clean_text examples that contained them, and be gone from topic_text.
ifb = ex[ex.apply(lambda r: "follow" in str(r.get("original_text", "")).lower(), axis=1)].head(5)
print("examples_containing_follow", len(ifb))
display(ifb[cols] if len(ifb) else ex[cols].head(3))
print("DONE preprocessing verification")


rows 36
columns ['tweet_id', 'source', 'keyword', 'original_lang', 'detected_lang', 'detect_status', 'nlp_status', 'original_text', 'clean_text', 'topic_text', 'clean_tokens', 'topic_tokens']
examples_containing_follow 5
DONE preprocessing verification


,original_text,clean_text,topic_text
0,"RT @Gainmutualsm: Retweet, Comment Let's GROW immediately ‼️:💛.💎🍃",rt retweet comment let grow immedi,retweet comment let grow immedi
1,"If you are a #Bitcoin maximalist, We want to follow you. Follow for follow. If you already follow me and I ain't, sorry. Drop a msg or even a DM.",bitcoin maximalist want follow follow follow alreadi follow sorri drop msg even dm,bitcoin maximalist want alreadi sorri drop msg even dm
2,"RT @bombsfall: ""The Last Of Us is the first good filmed video game adaptation"". That's cheating. The game is already a pretty grounded and…",rt last us first good film video game adapt cheat game alreadi pretti ground,last us first good film video game adapt cheat game alreadi pretti ground
3,Going into 2021 like... #followme #follow4follow #f4f #1980s #trending #Synthwave #synthwaveart #cyberpunk #cyberpunkart #Love #Art #artist #neon #neonnoir #light #arte #aesthetic #ａｅｓｔｈｅｔｉｃ #80s #westcoast #illgrammers #color #CobraKai #COBRAKAISEASON3 https://t.co/Wqjj5AjVdm,go like followm follow4follow f4f 1980 trend synthwav synthwaveart cyberpunk cyberpunkart love art artist neon neonnoir light art aesthet ａｅｓｔｈｅｔｉｃ 80 westcoast illgramm color cobrakai cobrakaiseason3,go like 1980 trend synthwav synthwaveart cyberpunk cyberpunkart love art artist neon neonnoir light art aesthet ａｅｓｔｈｅｔｉｃ 80 westcoast illgramm color cobrakai cobrakaiseason3
4,@BEFTrick Following all that follows @worldwideominie rapidly 107,follow follow rapidli,rapidli
5,"Uncovering the origins of Shandong, a vegetarian Chinese dish popular among Egyptians - and the Chinese-Egyptian community that invented it:\n\nhttps://t.co/6JD67wHQNY",uncov origin shandong vegetarian chines dish popular among egyptian chines egyptian commun invent,uncov origin shandong vegetarian chines dish popular among egyptian chines egyptian commun invent
6,@Stana_Katic #followmeandifollowyou @Stana_Katic @NathanFillion @Castle_ABC https://t.co/yHg3nTYsFU,followmeandifollowy,NaN
7,RT @THEL1VE_LABEL: [#WheeIn]\nWhee In 1st Full Album\n[IN the mood] 'In The Mood' MV TEASER #2\n \n✔ https://t.co/8tnwV0O7yy\n\n#휘인 #IN_the_mood…,rt wheein whee 1st full album mood mood mv teaser 휘인 in_the_mood,wheein whee 1st full album mood mood mv teaser 휘인 in_the_mood
8,"@fineboybeIla @wizkidayo 😂\n\nLet's follow each other, I am not a celebrity, I would follow you back, get me to 3k followers 😭😭😭😭😭",let follow celebr would follow back get 3k follow,let celebr would get 3k
9,@LeoOlebe hi. Could you follow back for DM,hi could follow back dm,hi could dm


,original_text,clean_text,topic_text
1,"If you are a #Bitcoin maximalist, We want to follow you. Follow for follow. If you already follow me and I ain't, sorry. Drop a msg or even a DM.",bitcoin maximalist want follow follow follow alreadi follow sorri drop msg even dm,bitcoin maximalist want alreadi sorri drop msg even dm
3,Going into 2021 like... #followme #follow4follow #f4f #1980s #trending #Synthwave #synthwaveart #cyberpunk #cyberpunkart #Love #Art #artist #neon #neonnoir #light #arte #aesthetic #ａｅｓｔｈｅｔｉｃ #80s #westcoast #illgrammers #color #CobraKai #COBRAKAISEASON3 https://t.co/Wqjj5AjVdm,go like followm follow4follow f4f 1980 trend synthwav synthwaveart cyberpunk cyberpunkart love art artist neon neonnoir light art aesthet ａｅｓｔｈｅｔｉｃ 80 westcoast illgramm color cobrakai cobrakaiseason3,go like 1980 trend synthwav synthwaveart cyberpunk cyberpunkart love art artist neon neonnoir light art aesthet ａｅｓｔｈｅｔｉｃ 80 westcoast illgramm color cobrakai cobrakaiseason3
4,@BEFTrick Following all that follows @worldwideominie rapidly 107,follow follow rapidli,rapidli
6,@Stana_Katic #followmeandifollowyou @Stana_Katic @NathanFillion @Castle_ABC https://t.co/yHg3nTYsFU,followmeandifollowy,NaN
8,"@fineboybeIla @wizkidayo 😂\n\nLet's follow each other, I am not a celebrity, I would follow you back, get me to 3k followers 😭😭😭😭😭",let follow celebr would follow back get 3k follow,let celebr would get 3k
